# 01 - Document Processing

**Goal:** turn PDF papers into small, clean text pieces ("chunks") that we can search later.

**Why this matters:** an LLM cannot read a 40-page paper at once, and a search engine works better on small pieces.
So the first job of any RAG system is: *read -> clean -> chunk -> save*.

**Steps in this notebook**
1. Find the PDFs
2. Read one PDF (look at the raw text)
3. Read all PDFs, page by page
4. Clean the text
5. Split into chunks (with overlap)
6. Experiment with chunk size
7. Save the chunks for the next notebook

> Before you start: put 2-3 PDF papers inside `data/papers/`

In [ ]:
# Install the one library we need to read PDFs (run once)
%pip install pypdf

## Step 1 - Find the PDFs

In [ ]:
from pathlib import Path   # Path makes file paths easy to work with
import json                # we will save our chunks as a JSON file
import re                  # re = regular expressions, used to clean text
from pypdf import PdfReader

# Folders (the notebook lives in notebooks/, so we go one level up with "..")
PAPERS_DIR = Path("../data/papers")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)   # create the folder if missing

# Find every .pdf file in the papers folder
pdf_files = sorted(PAPERS_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")
for f in pdf_files:
    print(" -", f.name)

## Step 2 - Read ONE PDF first

Always look at the raw text before building anything. PDFs are messy, and you want to see how messy.

In [ ]:
reader = PdfReader(pdf_files[0])          # open the first PDF
print("Number of pages:", len(reader.pages))

first_page = reader.pages[0].extract_text()   # get the text of page 1
print("\n--- First 600 characters of page 1 ---")
print(first_page[:600])

# TEACHING POINT: notice the strange line breaks, double spaces and split words.
# That is why we need a cleaning step.

## Step 3 - Read ALL PDFs, page by page

We keep the **file name** and **page number** with every piece of text.
Later this lets the assistant say *"this answer came from paper X, page 4"*.

In [ ]:
pages = []   # we will store one dictionary per page here

for pdf_path in pdf_files:
    reader = PdfReader(pdf_path)
    for page_number, page in enumerate(reader.pages, start=1):   # pages start at 1
        text = page.extract_text() or ""     # "or ''" protects us from empty pages
        if text.strip():                     # skip pages with no text (e.g. scanned images)
            pages.append({
                "source": pdf_path.name,
                "page": page_number,
                "text": text,
            })

print(f"Total pages with text: {len(pages)}")
print("Example page record keys:", list(pages[0].keys()))

## Step 4 - Clean the text

Two simple fixes that remove most of the mess:
- join words that were split across lines (`infor-\nmation` -> `information`)
- turn all the extra spaces and new lines into single spaces

In [ ]:
def clean_text(text):
    text = re.sub(r"-\n", "", text)    # join words broken with a hyphen at the line end
    text = re.sub(r"\s+", " ", text)   # many spaces/newlines -> one space
    return text.strip()                # remove spaces at the start and end

# Test the function on the raw first page from Step 2
print("BEFORE:", repr(first_page[:200]))
print()
print("AFTER :", repr(clean_text(first_page)[:200]))

# Apply it to every page
for p in pages:
    p["text"] = clean_text(p["text"])

## Step 5 - Split text into chunks

**Why chunks?** Small pieces = more precise search and shorter prompts for the LLM.

**Why overlap?** If a sentence is cut in half at the edge of a chunk, the overlap keeps the missing part
in the next chunk, so no meaning is lost.

```
chunk 1: [-------------]
chunk 2:           [-------------]
                   ^ overlap
```

In [ ]:
def chunk_text(text, chunk_size=800, overlap=150):
    # Safety check: overlap must be smaller than chunk_size, or the loop never ends
    assert overlap < chunk_size, "overlap must be smaller than chunk_size"

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])      # take one piece
        start += chunk_size - overlap       # move forward, but step back by 'overlap'
    return chunks

# Tiny demo so students can SEE the overlap
demo = "0123456789" * 5          # a 50-character string
for c in chunk_text(demo, chunk_size=20, overlap=5):
    print(c)
# Notice: each chunk starts with the last 5 characters of the previous one.

Now apply it to the real pages. Every chunk gets an `id` and keeps its `source` and `page`.

In [ ]:
CHUNK_SIZE = 800    # characters per chunk
OVERLAP = 150       # characters shared between neighbouring chunks

chunks = []
for p in pages:
    for piece in chunk_text(p["text"], CHUNK_SIZE, OVERLAP):
        if len(piece.strip()) > 50:     # ignore tiny leftover pieces
            chunks.append({
                "id": len(chunks),       # 0, 1, 2, ...
                "text": piece,
                "source": p["source"],
                "page": p["page"],
            })

print(f"Created {len(chunks)} chunks")
print("\nExample chunk:")
print(chunks[0])

## Step 6 - Experiment: how does chunk size change things?

Chunk size is the first setting you will tune in a RAG system.

In [ ]:
for size in [300, 500, 800, 1200, 2000]:
    total = 0
    for p in pages:
        total += len([c for c in chunk_text(p["text"], size, size // 5) if len(c.strip()) > 50])
    print(f"chunk_size={size:>5}  ->  {total} chunks")

# DISCUSS:
# - Small chunks  : precise, but may lose context
# - Large chunks  : more context, but the search gets less exact and prompts get longer

## Step 7 - Save the chunks for the next notebook

In [ ]:
output_path = PROCESSED_DIR / "chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Saved {len(chunks)} chunks to {output_path}")

## Exercises for students
1. Change `CHUNK_SIZE` to 400 and run again. How many chunks do you get now?
2. Print 3 random chunks. Do they make sense on their own?
3. Improve `clean_text` (for example remove page numbers or reference lists).
4. Instead of cutting by characters, try cutting by sentences or paragraphs.

**Next:** `02_embeddings.ipynb` - turning these chunks into numbers a computer can compare.